# Derivatives Matching Engine — PySpark / Databricks Production

**Document Reference:** Derivatives.docx (Business Requirements Document)

## Purpose
Production-grade PySpark implementation of the Derivatives Trade Matching Engine.
Replicates **exactly the same rules and logic** as the Pandas POC but optimised for:
- **4 M Axis × 20 M Finstore** rows
- **100+ columns** per dataset
- Databricks Runtime 16.4 LTS cluster (122 GB, 16 cores, 2–10 workers)

## Architecture (Best Practices Applied)
| Principle | Implementation |
|---|---|
| Match narrow, enrich wide | Core schemas (~15 cols) for matching; wide tables joined at the end |
| Candidates → Score → Resolve | All 15 BRD rules generate candidate edges; resolved via window ranking |
| No iterative pool removal | Priority + window ranking replaces repeated anti-joins |
| Block aggressively | Counterparty blocking (Strategy 1), amount-bucket blocking (Strategy 2) |
| Delta everywhere | CSV only as final export; optimizeWrite + autoCompact enabled |
| No Python UDFs | All derivations use native Spark SQL functions |
| Minimal .count() calls | ~6 essential counts vs ~20+ in original; arithmetic derivation where possible |
| Cluster-tuned config | 320 shuffle partitions, 256 MB broadcast, AQE + skew join |
| MEMORY_AND_DISK persist | Safe spill for large DataFrames at 4M+ row scale |

## Output Tables
| Table | Schema | Description |
|---|---|---|
| `matched_all_base` | Narrow (~15 cols) | Core match metadata (saved BEFORE enrichment) |
| `matched_all_enriched` | Wide (100+ cols) | Full enriched matches |
| `matched_brd_layer` | Narrow | BRD deterministic matches only |
| `matched_greedy_layer` | Narrow | Greedy probabilistic matches only |
| `unmatched_axis_base` | Narrow | Unmatched Axis core |
| `unmatched_axis_enriched` | Wide | Unmatched Axis full |
| `unmatched_finstore_base` | Narrow | Unmatched Finstore core |
| `unmatched_finstore_enriched` | Wide | Unmatched Finstore full |

## Layers
- **Layer 1:** BRD Deterministic Rules (15 total: 3 SOPHIS + 10 OTC + 2 ETD)
- **Layer 2:** Greedy / Probabilistic Matching
  - Strategy 1: Amount + Counterparty (1% tolerance)
  - Strategy 2: Amount-only strict (0.1% tolerance)

## Scope Note
SOPHIS and DELTA1 systems are excluded by default.
Their trade IDs do not exist in Finstore — this is a data mapping issue, not a matching algorithm issue.

**Date:** 2026-02-23 (v4.1 — PySpark Production, cluster-optimised)

---
## 🔹 Section 1: Spark Session & Configuration

In [ ]:
# ============================================================
# SPARK SESSION & IMPORTS
# ============================================================
# On Databricks the SparkSession is pre-created as `spark`.
# For local testing uncomment the builder block below.

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window
from pyspark import StorageLevel
from functools import reduce
from datetime import datetime
import logging

# --- Uncomment for local / non-Databricks execution ---
# spark = (
#     SparkSession.builder
#     .master("local[*]")
#     .appName("DerivativesMatchingEngine")
#     .config("spark.sql.adaptive.enabled", "true")
#     .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
#     .config("spark.sql.adaptive.skewJoin.enabled", "true")
#     .config("spark.sql.shuffle.partitions", "200")
#     .config("spark.driver.memory", "8g")
#     .getOrCreate()
# )

# ============================================================
# CLUSTER-TUNED SETTINGS — 122 GB, 16 cores, 2-10 workers
# Databricks Runtime 16.4 LTS
# ============================================================

# --- AQE (Adaptive Query Execution) ---
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.localShuffleReader.enabled", "true")

# Shuffle partitions: ~2-3× total cores across cluster
# With 10 workers × 16 cores = 160 cores → 320 partitions is a good start.
# AQE will coalesce down if too many are empty.
spark.conf.set("spark.sql.shuffle.partitions", "320")

# Broadcast threshold: 256 MB (Axis core ~4M × 11 cols ≈ 200-400 MB,
# if it fits, Spark will auto-broadcast the smaller side of BRD joins)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(256 * 1024 * 1024))

# Optimize large shuffles — tungsten sort-based shuffle for better memory use
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "128m")

# Reduce small file problem on Delta writes
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.autoCompact.enabled", "true")

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

print(f"Spark version: {spark.version}")
print(f"AQE enabled: {spark.conf.get('spark.sql.adaptive.enabled')}")
print(f"Skew join: {spark.conf.get('spark.sql.adaptive.skewJoin.enabled')}")
print(f"Shuffle partitions: {spark.conf.get('spark.sql.shuffle.partitions')}")
print(f"Broadcast threshold: {int(spark.conf.get('spark.sql.autoBroadcastJoinThreshold'))/(1024*1024):.0f} MB")
print(f"Delta optimizeWrite: {spark.conf.get('spark.databricks.delta.optimizeWrite.enabled')}")
print(f"Delta autoCompact: {spark.conf.get('spark.databricks.delta.autoCompact.enabled')}")

In [ ]:
# ============================================================
# SCOPE CONFIGURATION
# ============================================================

# Set to True to exclude SOPHIS and DELTA1 systems from scope
# These systems' trade IDs don't exist in Finstore (data mapping issue)
EXCLUDE_SOPHIS_DELTA1 = True

# Out-of-scope systems (when EXCLUDE_SOPHIS_DELTA1 = True)
OUT_OF_SCOPE_SYSTEMS = [
    "SOPHIS-LONDON", "SOPHIS-NEWYORK", "SOPHIS-TOKYO", "SOPHISFX-LONDON",
    "DELTA1-LONDON", "DELTA1-NEWYORK"
]

# Greedy matching parameters
GREEDY_TOLERANCE_PCT = 0.01     # 1% for Strategy 1 (Amount + Counterparty)
STRICT_TOLERANCE_PCT = 0.001    # 0.1% for Strategy 2 (Amount only)
BUCKET_SIZE = 1000              # Amount bucket size for Strategy 2 blocking

# ============================================================
# LINEAGE & AUDITABILITY  (Best Practice §6)
# ============================================================
# Every match row will carry these for full traceability.
import uuid

RUN_ID = str(uuid.uuid4())                           # unique per execution
BATCH_ID = datetime.now().strftime("%Y%m%d_%H%M%S")  # human-readable batch
RUN_TIMESTAMP = datetime.now().isoformat()
RULE_VERSION = "v4.0-pyspark"                         # bump on logic changes

# ============================================================
# FILE PATHS — Adjust for Databricks / ADLS / DBFS
# ============================================================

# For Databricks: use dbfs:/ or abfss:// paths
# For local: use file:///path/to/files
INPUT_DIR = "/mnt/data/derivatives"           # <-- adjust to your mount / path
INPUT_FILE_AXIS = f"{INPUT_DIR}/axis_sample_poc.csv"
INPUT_FILE_FINSTORE = f"{INPUT_DIR}/finstore_sample_poc.csv"
SDS_MAPPING_FILE = f"{INPUT_DIR}/sds_book_mapping.csv"

OUTPUT_DIR = f"{INPUT_DIR}/matching_results"

print("SCOPE CONFIGURATION:")
print(f"  EXCLUDE_SOPHIS_DELTA1: {EXCLUDE_SOPHIS_DELTA1}")
if EXCLUDE_SOPHIS_DELTA1:
    print(f"  Excluded systems: {', '.join(OUT_OF_SCOPE_SYSTEMS)}")
print(f"\nGREEDY MATCHING PARAMETERS:")
print(f"  Strategy 1 tolerance: {GREEDY_TOLERANCE_PCT * 100}%")
print(f"  Strategy 2 tolerance: {STRICT_TOLERANCE_PCT * 100}%")
print(f"  Bucket size: {BUCKET_SIZE}")
print(f"\nLINEAGE:")
print(f"  run_id:       {RUN_ID}")
print(f"  batch_id:     {BATCH_ID}")
print(f"  rule_version: {RULE_VERSION}")
print(f"\nPATHS:")
print(f"  Input:  {INPUT_DIR}")
print(f"  Output: {OUTPUT_DIR}")

---
## 🔹 Section 2: BRD Constants & System Classifications

In [ ]:
# ============================================================
# BRD SECTION: System Classification Lists
# ============================================================

# OTC Systems (from BRD table)
OTC_SYSTEMS = [
    "ALD-SF", "BARXFX-PD-ASIAPAC", "BARXFX-PD-LONDON", "BARXFETS",
    "BARXFX-TS-CASH-ASIAPAC", "BARXFX-TS-CASH-LONDON", "DELTA1-LONDON",
    "DELTA1-NEWYORK", "IFLOW-AMERICAS", "IFLOW-ASIAPACIFIC", "IFLOW-EUROPE",
    "FISSFX-NEWYORK", "FISS-LONDON", "FISS-TOKYO", "GCD-NEWYORK",
    "IMPACT-NEWYORK", "OPENLINK-LONDON", "OPTISRD-MEXICO", "SABS-NEWYORK",
    "SOPHISFX-LONDON", "SOPHIS-LONDON", "SOPHIS-NEWYORK", "SOPHIS-TOKYO",
    "SUMMIT-LONDON", "SYNFINY-CFD-NONCASH", "SYETR-NEWYORK-MBS"
]

# ETD Systems (from BRD table)
ETD_SYSTEMS = [
    "BPS222-OPT-NEWYORK", "BPS224-OPT-NEWYORK", "ODH-DOLPHIN-INDIA",
    "ODH-CMI-CPM-LONDON", "ODH-GMI-HONGKONG", "ODH-GMI-LONDON",
    "ODH-GMI-LONDON-BDT", "ODH-GMI-NEWYORK", "ODH-GMI-SINGAPORE",
    "ODH-ISTAR-TOKYO", "OPTISFUT-MEXICO"
]

# SOPHIS Systems
SOPHIS_SYSTEMS = [
    "SOPHISFX-LONDON", "SOPHIS-LONDON", "SOPHIS-TOKYO", "SOPHIS-NEWYORK"
]

# DELTA1 Systems
DELTA1_SYSTEMS = ["DELTA1-LONDON", "DELTA1-NEWYORK"]

# Amount column names
AXIS_AMOUNT_COL = "SACCRMTM"
FIN_AMOUNT_COL = "gbpequivalentamount"

print("BRD CONSTANTS LOADED")
print(f"  OTC Systems:    {len(OTC_SYSTEMS)}")
print(f"  ETD Systems:    {len(ETD_SYSTEMS)}")
print(f"  SOPHIS Systems: {len(SOPHIS_SYSTEMS)}")
print(f"  DELTA1 Systems: {len(DELTA1_SYSTEMS)}")

---
## 🔹 Section 3: Matching Rule Definitions (All 15 BRD Rules)

Rules are defined as lightweight dictionaries instead of dataclasses.
Each rule specifies:
- `priority`: global waterfall order (1 = highest priority)
- `category`: SOPHIS / OTC / ETD
- `brd_priority`: priority within category
- `axis_keys` / `fin_keys`: column(s) to join on
- `filter_sophis_only`: whether to restrict Axis to SOPHIS systems
- `requires_derived_masterbook`: skipped if DerivedMasterbookId is empty

In [ ]:
# ============================================================
# ALL 15 BRD MATCHING RULES
# ============================================================
# Exact same rules as the Pandas notebook, expressed as dicts
# for easy iteration in Spark.

MATCHING_RULES = [
    # ---- SOPHIS-specific rules (Priority 1-3) ----
    dict(
        priority=1, category="SOPHIS", brd_priority=1,
        description="DerivedSophisId <> fissnumber",
        axis_keys=["DerivedSophisId"],
        fin_keys=["fissnumber"],
        filter_sophis_only=True,
        requires_derived_masterbook=False,
    ),
    dict(
        priority=2, category="SOPHIS", brd_priority=2,
        description="DerivedSophisId + BookId <> fissnumber + tradingsystembook",
        axis_keys=["DerivedSophisId", "BookId"],
        fin_keys=["fissnumber", "tradingsystembook"],
        filter_sophis_only=True,
        requires_derived_masterbook=False,
    ),
    dict(
        priority=3, category="SOPHIS", brd_priority=3,
        description="DerivedSophisId <> tradeid",
        axis_keys=["DerivedSophisId"],
        fin_keys=["tradeid"],
        filter_sophis_only=True,
        requires_derived_masterbook=False,
    ),

    # ---- OTC rules (Priority 4-13) ----
    dict(
        priority=4, category="OTC", brd_priority=1,
        description="SourceSystemTradeId <> tradeid",
        axis_keys=["SourceSystemTradeId"],
        fin_keys=["tradeid"],
        filter_sophis_only=False,
        requires_derived_masterbook=False,
    ),
    dict(
        priority=5, category="OTC", brd_priority=2,
        description="SourceSystemTradeId + DerivedMasterbookId <> tradeid + masterbookid",
        axis_keys=["SourceSystemTradeId", "DerivedMasterbookId"],
        fin_keys=["tradeid", "masterbookid"],
        filter_sophis_only=False,
        requires_derived_masterbook=True,
    ),
    dict(
        priority=6, category="OTC", brd_priority=3,
        description="SourceSystemTradeId <> alternatetradeid1",
        axis_keys=["SourceSystemTradeId"],
        fin_keys=["alternatetradeid1"],
        filter_sophis_only=False,
        requires_derived_masterbook=False,
    ),
    dict(
        priority=7, category="OTC", brd_priority=4,
        description="SourceSystemTradeId + DerivedMasterbookId <> alternatetradeid1 + masterbookid",
        axis_keys=["SourceSystemTradeId", "DerivedMasterbookId"],
        fin_keys=["alternatetradeid1", "masterbookid"],
        filter_sophis_only=False,
        requires_derived_masterbook=True,
    ),
    dict(
        priority=8, category="OTC", brd_priority=5,
        description="DerivedSophisId <> fissnumber",
        axis_keys=["DerivedSophisId"],
        fin_keys=["fissnumber"],
        filter_sophis_only=False,
        requires_derived_masterbook=False,
    ),
    dict(
        priority=9, category="OTC", brd_priority=6,
        description="DerivedSophisId + BookId <> fissnumber + tradingsystembook",
        axis_keys=["DerivedSophisId", "BookId"],
        fin_keys=["fissnumber", "tradingsystembook"],
        filter_sophis_only=False,
        requires_derived_masterbook=False,
    ),
    dict(
        priority=10, category="OTC", brd_priority=7,
        description="DerivedSophisId <> tradeid",
        axis_keys=["DerivedSophisId"],
        fin_keys=["tradeid"],
        filter_sophis_only=False,
        requires_derived_masterbook=False,
    ),
    dict(
        priority=11, category="OTC", brd_priority=8,
        description="DerivedDelta1Id <> tradeid",
        axis_keys=["DerivedDelta1Id"],
        fin_keys=["tradeid"],
        filter_sophis_only=False,
        requires_derived_masterbook=False,
    ),
    dict(
        priority=12, category="OTC", brd_priority=9,
        description="SourceSystemTradeId <> alternatetradeid2",
        axis_keys=["SourceSystemTradeId"],
        fin_keys=["alternatetradeid2"],
        filter_sophis_only=False,
        requires_derived_masterbook=False,
    ),
    dict(
        priority=13, category="OTC", brd_priority=10,
        description="SourceSystemTradeId + DerivedMasterbookId <> alternatetradeid2 + masterbookid",
        axis_keys=["SourceSystemTradeId", "DerivedMasterbookId"],
        fin_keys=["alternatetradeid2", "masterbookid"],
        filter_sophis_only=False,
        requires_derived_masterbook=True,
    ),

    # ---- ETD rules (Priority 14-15) ----
    dict(
        priority=14, category="ETD", brd_priority=1,
        description="SourceSystemInstrumentId + DerivedMasterbookId <> instrumentid + masterbookid",
        axis_keys=["SourceSystemInstrumentId", "DerivedMasterbookId"],
        fin_keys=["instrumentid", "masterbookid"],
        filter_sophis_only=False,
        requires_derived_masterbook=True,
    ),
    dict(
        priority=15, category="ETD", brd_priority=2,
        description="SourceSystemInstrumentId <> instrumentid",
        axis_keys=["SourceSystemInstrumentId"],
        fin_keys=["instrumentid"],
        filter_sophis_only=False,
        requires_derived_masterbook=False,
    ),
]

sophis_count = sum(1 for r in MATCHING_RULES if r["category"] == "SOPHIS")
otc_count = sum(1 for r in MATCHING_RULES if r["category"] == "OTC")
etd_count = sum(1 for r in MATCHING_RULES if r["category"] == "ETD")

print(f"SOPHIS rules: {sophis_count}")
print(f"OTC rules:    {otc_count}")
print(f"ETD rules:    {etd_count}")
print(f"TOTAL:        {len(MATCHING_RULES)} rules")

---
## 🔹 Section 4: Load Data (Bronze Layer)

In [ ]:
# ============================================================
# BRONZE LAYER — Raw Ingestion
# ============================================================
# For production: read from Delta tables instead of CSV.
# Example: spark.read.format("delta").load("dbfs:/mnt/...")
#
# ⚡ PERF: No .count() here — counts are deferred to cache materialisation
#    in Section 7 to avoid triggering 3 extra Spark jobs at load time.

logger.info("Loading Axis data...")
df_axis_full = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(INPUT_FILE_AXIS)
)
# Bronze metadata (Best Practice §2A — append-only lineage)
df_axis_full = (
    df_axis_full
    .withColumn("_ingest_timestamp", F.lit(RUN_TIMESTAMP))
    .withColumn("_source_file", F.lit(INPUT_FILE_AXIS))
    .withColumn("_batch_id", F.lit(BATCH_ID))
)

logger.info("Loading Finstore data...")
df_finstore_full = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(INPUT_FILE_FINSTORE)
)
# Bronze metadata
df_finstore_full = (
    df_finstore_full
    .withColumn("_ingest_timestamp", F.lit(RUN_TIMESTAMP))
    .withColumn("_source_file", F.lit(INPUT_FILE_FINSTORE))
    .withColumn("_batch_id", F.lit(BATCH_ID))
)

# Load SDS mapping if available (small table — good broadcast candidate)
try:
    df_sds_mapping = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(SDS_MAPPING_FILE)
    )
    sds_available = True
    logger.info("SDS mapping loaded")
except Exception:
    df_sds_mapping = None
    sds_available = False
    logger.info("SDS mapping not available — DerivedMasterbookId will be empty")

print(f"\n{'='*80}")
print("DATA LOADED (BRONZE) — counts deferred to cache materialisation")
print(f"{'='*80}")
print(f"Axis columns:    {len(df_axis_full.columns)}")
print(f"Finstore columns:{len(df_finstore_full.columns)}")
print(f"SDS Mapping:     {'Available' if sds_available else 'NOT AVAILABLE'}")

---
## 🔹 Section 5: Scope Exclusion (SOPHIS / DELTA1)

In [ ]:
# ============================================================
# SCOPE EXCLUSION
# ============================================================

print(f"\n{'='*80}")
print("SCOPE EXCLUSION")
print(f"{'='*80}")

if EXCLUDE_SOPHIS_DELTA1:
    df_axis_excluded = df_axis_full.filter(
        F.col("SourceSystemName").isin(OUT_OF_SCOPE_SYSTEMS)
    )
    df_axis = df_axis_full.filter(
        ~F.col("SourceSystemName").isin(OUT_OF_SCOPE_SYSTEMS)
    )

    # ── Exclusion breakdown by system ────────────────────────────────────
    # One Spark job to show exactly how many trades each system contributes.
    print("\n*** SOPHIS/DELTA1 SCOPE EXCLUSION ACTIVE ***\n")
    print("Excluded systems breakdown:")
    excluded_breakdown = (
        df_axis_excluded
        .groupBy("SourceSystemName")
        .count()
        .orderBy(F.col("count").desc())
        .collect()
    )
    for row in excluded_breakdown:
        print(f"  {row['SourceSystemName']:<25s} : {row['count']:>10,}")

    total_excluded = sum(row['count'] for row in excluded_breakdown)
    total_full = df_axis_full.count()
    print(f"  {'─'*39}")
    print(f"  {'TOTAL EXCLUDED':<25s} : {total_excluded:>10,}")
    print(f"  {'REMAINING IN-SCOPE':<25s} : {total_full - total_excluded:>10,}")
    print(f"  {'ORIGINAL (full)':<25s} : {total_full:>10,}")
    print(f"  Excluded %: {total_excluded/total_full*100:.1f}%")
else:
    df_axis = df_axis_full
    df_axis_excluded = spark.createDataFrame([], df_axis_full.schema)
    print("All systems in scope.")

---
## 🔹 Section 6: Pre-Reconciliation Derivations (Silver Layer)

All derivations use **native Spark SQL functions** — no Python UDFs.
- `DerivedSophisId`: 3rd part of hyphen-split `SourceSystemTradeId` (SOPHIS systems)
- `DerivedDelta1Id`: 3rd part of hyphen-split `SourceSystemTradeId` (DELTA1 systems)
- `ReconSubProduct`: ETD / OTC / OTC-Default classification
- `DerivedMasterbookId`: from SDS mapping (if available)

In [ ]:
# ============================================================
# SILVER LAYER — Derivations & Key Preparation
# ============================================================
# All done with native Spark SQL — no Python UDFs.
# ⚡ PERF: No .count() calls — derivations are lazy transformations.

# --- Stable surrogate IDs (monotonically_increasing_id is unique per run) ---
df_axis = df_axis.withColumn("axis_id", F.monotonically_increasing_id())
df_finstore = df_finstore_full.withColumn("fin_id", F.monotonically_increasing_id())

# --- ReconSubProduct classification ---
df_axis = df_axis.withColumn(
    "ReconSubProduct",
    F.when(F.col("SourceSystemName").isin(ETD_SYSTEMS), F.lit("ETD"))
     .when(F.col("SourceSystemName").isin(OTC_SYSTEMS), F.lit("OTC"))
     .otherwise(F.lit("OTC-Default"))
)

# --- DerivedSophisId: extract 3rd part of hyphen-separated SourceSystemTradeId ---
# Native Spark: split() + element_at() — no UDF needed
df_axis = df_axis.withColumn(
    "DerivedSophisId",
    F.when(
        F.col("SourceSystemName").isin(SOPHIS_SYSTEMS) &
        F.col("SourceSystemTradeId").isNotNull() &
        (F.size(F.split(F.col("SourceSystemTradeId"), "-")) >= 3),
        F.element_at(F.split(F.col("SourceSystemTradeId"), "-"), 3)
    ).otherwise(F.lit(""))
)

# --- DerivedDelta1Id: same extraction for DELTA1 systems ---
df_axis = df_axis.withColumn(
    "DerivedDelta1Id",
    F.when(
        F.col("SourceSystemName").isin(DELTA1_SYSTEMS) &
        F.col("SourceSystemTradeId").isNotNull() &
        (F.size(F.split(F.col("SourceSystemTradeId"), "-")) >= 3),
        F.element_at(F.split(F.col("SourceSystemTradeId"), "-"), 3)
    ).otherwise(F.lit(""))
)

# --- DerivedMasterbookId placeholder (requires SDS mapping) ---
# TODO: When SDS is available, do a broadcast join here:
# df_axis = df_axis.join(broadcast(df_sds_mapping), on=<key>, how="left")
df_axis = df_axis.withColumn("DerivedMasterbookId", F.lit(""))

# --- Ensure amount columns are numeric ---
df_axis = df_axis.withColumn(AXIS_AMOUNT_COL, F.col(AXIS_AMOUNT_COL).cast("double"))
df_finstore = df_finstore.withColumn(FIN_AMOUNT_COL, F.col(FIN_AMOUNT_COL).cast("double"))

# --- Ensure string join columns are trimmed ---
string_cols_axis = [
    "SourceSystemTradeId", "BookId", "DerivedSophisId", "DerivedDelta1Id",
    "DerivedMasterbookId", "SourceSystemInstrumentId", "CounterpartyId"
]
for c in string_cols_axis:
    if c in df_axis.columns:
        df_axis = df_axis.withColumn(c, F.trim(F.col(c).cast("string")))

string_cols_fin = [
    "tradeid", "alternatetradeid1", "alternatetradeid2", "masterbookid",
    "tradingsystembook", "fissnumber", "instrumentid", "counterpartyid"
]
for c in string_cols_fin:
    if c in df_finstore.columns:
        df_finstore = df_finstore.withColumn(c, F.trim(F.col(c).cast("string")))

print("PRE-RECONCILIATION DERIVATIONS DEFINED (lazy — no Spark jobs triggered)")
print("Derivations will materialise when core tables are cached in the next section.")

---
## 🔹 Section 7: Core / Wide Split (Match Narrow, Enrich Wide)

**Critical performance optimisation:** only carry the ~15 columns needed for matching.
The remaining 100+ columns stay in `_wide` tables and are joined back at the end.

In [ ]:
# ============================================================
# CORE / WIDE SPLIT — minimise shuffle payload
# ============================================================
# ⚡ PERF: These .count() calls are INTENTIONAL — they materialise the cache.
#    This is the single point where all deferred counts are resolved.
#    Uses MEMORY_AND_DISK to spill gracefully if 4M+ rows don't fit in RAM.

# Axis core: only columns needed for matching
AXIS_CORE_COLS = [
    "axis_id", "SourceSystemName", "SourceSystemTradeId", "BookId",
    "CounterpartyId", AXIS_AMOUNT_COL, "SourceSystemInstrumentId",
    "DerivedSophisId", "DerivedDelta1Id", "DerivedMasterbookId",
    "ReconSubProduct",
]
# Filter to columns that actually exist
AXIS_CORE_COLS = [c for c in AXIS_CORE_COLS if c in df_axis.columns]

axis_core = df_axis.select(AXIS_CORE_COLS)
axis_wide = df_axis  # keep full DataFrame; we join by axis_id at the end

# Finstore core: only columns needed for matching
FIN_CORE_COLS = [
    "fin_id", "tradeid", "alternatetradeid1", "alternatetradeid2",
    "masterbookid", "tradingsystembook", "fissnumber", "instrumentid",
    "counterpartyid", FIN_AMOUNT_COL,
]
FIN_CORE_COLS = [c for c in FIN_CORE_COLS if c in df_finstore.columns]

fin_core = df_finstore.select(FIN_CORE_COLS)
fin_wide = df_finstore  # keep full DataFrame; we join by fin_id at the end

# Cache core tables with MEMORY_AND_DISK — they are reused across all 15 rules + greedy
axis_core = axis_core.persist(StorageLevel.MEMORY_AND_DISK)
fin_core = fin_core.persist(StorageLevel.MEMORY_AND_DISK)

# Materialise cache + get counts (single Spark job per DF)
ORIGINAL_AXIS_COUNT = axis_core.count()
ORIGINAL_FINSTORE_COUNT = fin_core.count()

# Derive excluded count arithmetically (no extra Spark job)
if EXCLUDE_SOPHIS_DELTA1:
    # We need the full count — but we can get it from df_axis_full cheaply
    # since it's a simple CSV read that was already scanned above.
    # Use a single quick count on the excluded DF instead.
    ORIGINAL_AXIS_COUNT_FULL = ORIGINAL_AXIS_COUNT  # will be updated below
    excluded_count = 0  # placeholder
    # Compute full count once
    _full_axis_count = df_axis_full.count()
    ORIGINAL_AXIS_COUNT_FULL = _full_axis_count
    excluded_count = ORIGINAL_AXIS_COUNT_FULL - ORIGINAL_AXIS_COUNT
else:
    ORIGINAL_AXIS_COUNT_FULL = ORIGINAL_AXIS_COUNT
    excluded_count = 0

print(f"\n{'='*80}")
print("CORE TABLES CACHED — ALL DEFERRED COUNTS RESOLVED")
print(f"{'='*80}")
print(f"Axis (full):     {ORIGINAL_AXIS_COUNT_FULL:,} records")
print(f"Axis (in-scope): {ORIGINAL_AXIS_COUNT:,} records ({len(AXIS_CORE_COLS)} core cols)")
if EXCLUDE_SOPHIS_DELTA1:
    print(f"Axis (excluded): {excluded_count:,} records")
print(f"Finstore:        {ORIGINAL_FINSTORE_COUNT:,} records ({len(FIN_CORE_COLS)} core cols)")
print(f"\nAxis wide cols:    {len(df_axis.columns)}")
print(f"Finstore wide cols:{len(df_finstore.columns)}")

---
## ✅ LAYER 1: BRD DETERMINISTIC MATCHING (15 Rules)

### Architecture: Candidates → Score → Resolve

Instead of executing rules one-by-one and mutating pools (13+ shuffles),
we:
1. **Generate candidate edges** for each rule (one join per rule, narrow schema)
2. **Union all candidates** into a single DataFrame
3. **Resolve: best rule per axis** (waterfall parity), keeping all within-rule matches

Each axis keeps only its **highest-priority (lowest number) rule's** matches,
replicating the Pandas sequential waterfall where matched trades are removed
from the pool after each rule. Within that winning rule, all inner-join rows
are preserved (1 axis → N finstores if the key hits multiple records).
Pool removal uses distinct axis/fin IDs (identical to Pandas `set(unique())`).

---
### 🔹 Section 8: Candidate Generation Function

In [ ]:
# ============================================================
# CANDIDATE GENERATION — one function for all BRD rules
# ============================================================
# ⚡ PERF: Pre-compute the DerivedMasterbookId flag ONCE instead of
#    calling .limit(1).count() inside every rule (was 5 extra Spark jobs).

# Check once: does ANY Axis row have a non-empty DerivedMasterbookId?
HAS_DERIVED_MASTERBOOK = (
    axis_core
    .filter((F.col("DerivedMasterbookId").isNotNull()) & (F.col("DerivedMasterbookId") != ""))
    .limit(1)
    .count() > 0
)
print(f"DerivedMasterbookId populated: {HAS_DERIVED_MASTERBOOK}")

# Empty candidate schema (reused across skipped rules — created once)
_EMPTY_CANDIDATE_SCHEMA = T.StructType([
    T.StructField("axis_id", T.LongType()),
    T.StructField("fin_id", T.LongType()),
    T.StructField("priority", T.IntegerType()),
    T.StructField("category", T.StringType()),
    T.StructField("brd_priority", T.IntegerType()),
    T.StructField("description", T.StringType()),
    T.StructField("amount_diff", T.DoubleType()),
    T.StructField("amount_rel_diff", T.DoubleType()),
    T.StructField("key_strength", T.IntegerType()),
])


def build_candidates_for_rule(
    axis_df: DataFrame,
    fin_df: DataFrame,
    rule: dict,
) -> DataFrame:
    """
    Generate candidate match edges for a single BRD rule.
    Returns a DataFrame with:
        axis_id, fin_id, priority, category, brd_priority,
        description, amount_diff, amount_rel_diff, key_strength
    """
    priority = rule["priority"]
    axis_keys = rule["axis_keys"]
    fin_keys = rule["fin_keys"]

    # --- System filter ---
    a = axis_df
    if rule["filter_sophis_only"]:
        a = a.filter(F.col("SourceSystemName").isin(SOPHIS_SYSTEMS))

    # --- Skip if DerivedMasterbookId required but not populated ---
    # ⚡ Uses pre-computed flag — NO Spark job triggered here.
    if rule["requires_derived_masterbook"] and not HAS_DERIVED_MASTERBOOK:
        return spark.createDataFrame([], _EMPTY_CANDIDATE_SCHEMA)

    # --- Project to narrow schema ---
    a_cols = list(dict.fromkeys(["axis_id", AXIS_AMOUNT_COL] + axis_keys))  # dedup
    a_sel = a.select([F.col(c) for c in a_cols])

    f_cols = list(dict.fromkeys(["fin_id", FIN_AMOUNT_COL] + fin_keys))  # dedup
    f_sel = fin_df.select([F.col(c) for c in f_cols])

    # --- Filter out invalid keys (empty, 'nan', containing '$') ---
    for k in axis_keys:
        a_sel = a_sel.filter(
            (F.col(k).isNotNull()) &
            (F.col(k) != "") &
            (F.col(k) != "nan") &
            (~F.col(k).contains("$"))
        )
    for k in fin_keys:
        f_sel = f_sel.filter(
            (F.col(k).isNotNull()) &
            (F.col(k) != "") &
            (F.col(k) != "nan") &
            (~F.col(k).contains("$"))
        )

    # --- Rename keys to common join names (k0, k1, ...) ---
    for i, ak in enumerate(axis_keys):
        a_sel = a_sel.withColumnRenamed(ak, f"_jk{i}")
    for i, fk in enumerate(fin_keys):
        f_sel = f_sel.withColumnRenamed(fk, f"_jk{i}")

    join_cols = [f"_jk{i}" for i in range(len(axis_keys))]

    # --- Join ---
    joined = a_sel.join(f_sel, on=join_cols, how="inner")

    # --- Compute amount_diff (tiebreaker) ---
    joined = joined.withColumn(
        "amount_diff",
        F.abs(F.col(AXIS_AMOUNT_COL) - F.col(FIN_AMOUNT_COL))
    )

    # --- Scoring metrics (Best Practice §2D) ---
    joined = joined.withColumn(
        "amount_rel_diff",
        F.col("amount_diff") / F.greatest(F.abs(F.col(AXIS_AMOUNT_COL)), F.lit(1e-9))
    )
    key_strength = len(axis_keys)
    joined = joined.withColumn("key_strength", F.lit(key_strength).cast("int"))

    # --- Output standardised candidate schema ---
    return joined.select(
        F.col("axis_id"),
        F.col("fin_id"),
        F.lit(priority).cast("int").alias("priority"),
        F.lit(rule["category"]).alias("category"),
        F.lit(rule["brd_priority"]).cast("int").alias("brd_priority"),
        F.lit(rule["description"]).alias("description"),
        F.col("amount_diff"),
        F.col("amount_rel_diff"),
        F.col("key_strength"),
    )


print("Candidate generation function defined.")

---
### 🔹 Section 9: Generate All BRD Candidates

In [ ]:
# ============================================================
# GENERATE CANDIDATE EDGES FOR ALL 15 BRD RULES
# ============================================================
# ⚡ PERF: All 15 rules are lazy transformations. The union is also lazy.
#    We persist with MEMORY_AND_DISK but do NOT call .count() here.
#    The materialisation happens in the next cell (resolve_matches)
#    which must scan the data anyway.

print(f"\n{'='*80}")
print("LAYER 1: GENERATING BRD CANDIDATES")
print(f"{'='*80}")

candidate_dfs = []

for rule in MATCHING_RULES:
    cand = build_candidates_for_rule(axis_core, fin_core, rule)
    candidate_dfs.append(cand)
    print(f"  Rule {rule['priority']:2d} ({rule['category']:6s} #{rule['brd_priority']}): "
          f"{rule['description']}")

# Union all candidate edges (lazy — no Spark job)
candidates_layer1 = reduce(DataFrame.unionByName, candidate_dfs)

# Persist — materialisation will happen when resolve_matches reads it
candidates_layer1 = candidates_layer1.persist(StorageLevel.MEMORY_AND_DISK)

print(f"\nCandidates unioned (lazy). Will materialise during resolution.")

---
### 🔹 Section 10: BRD Match Resolution — Waterfall Parity (Best Rule per Axis)

Replicates the Pandas **sequential waterfall** semantics:

- Pandas runs rules 1→15 sequentially: after each rule, matched axis AND
  finstore IDs are **removed from the pool**.  So each axis matches via
  **at most one rule** (its first/highest-priority hit).
- **Within** that winning rule, `pd.merge(inner)` CAN produce multiple
  rows if the key hits several finstore records (many-to-many).
- PySpark runs all 15 rules against the full pool in parallel, then
  unions them.  To replicate the waterfall we keep, **for each axis_id,
  only the rows from its best (lowest) priority rule**.  All finstore
  matches within that rule are preserved (many-to-many within a rule).
- **Pool removal** uses the distinct set of matched `axis_id` and `fin_id`
  values — identical to Pandas `set(matched[idx].unique())`.
- `resolve_matches()` (single best fin per axis) is used **only** by
  Greedy layers, matching Pandas `nsmallest(1, "amount_diff")`.

In [ ]:
# ============================================================
# BRD MATCH RESOLUTION — Waterfall Parity (Best Rule per Axis)
# ============================================================
# Replicates the Pandas sequential waterfall:
#   Pandas runs rules 1→15 with pool removal after each rule, so
#   each axis matches via AT MOST ONE rule (its first hit).
#   Within that rule, pd.merge(inner) can produce multiple rows
#   (1 axis key → N finstores).
#
# PySpark runs all 15 rules against full pools (parallel), then
# unions them.  To replicate the waterfall:
#   1. For each axis_id, find its BEST (lowest) priority.
#   2. Keep ALL rows for that axis at that priority level.
# This preserves within-rule many-to-many while preventing
# cross-rule explosion (1 axis matching via rules 1, 3, 5, 7…).
#
# Pool removal uses DISTINCT axis_id / fin_id sets — identical to
# Pandas set(matched[idx].unique()).

# ── resolve_matches: used ONLY by Greedy layers ─────────────────────────
def resolve_matches(candidates: DataFrame) -> DataFrame:
    """
    For each axis_id, pick the single best fin_id candidate.
    Used by Greedy layers only (Pandas greedy also picks 1 best per axis
    via nsmallest(1, 'amount_diff')).

    Ranking criteria (in order):
        1. lowest priority
        2. highest key_strength
        3. smallest amount_diff
        4. smallest fin_id  (deterministic / stable)
    """
    has_key_strength = "key_strength" in candidates.columns

    ordering = [F.col("priority").asc()]
    if has_key_strength:
        ordering.append(F.col("key_strength").desc())
    ordering.append(F.col("amount_diff").asc_nulls_last())

    w_axis = Window.partitionBy("axis_id").orderBy(*ordering, F.col("fin_id").asc())
    resolved = (
        candidates
        .withColumn("_rn_axis", F.row_number().over(w_axis))
        .filter(F.col("_rn_axis") == 1)
        .drop("_rn_axis")
    )
    return resolved


# ── BRD: keep best-rule per axis, all fin matches within that rule ──────
print("BRD resolution: waterfall parity (best rule per axis, many-to-many within rule)...")

# Step 1: For each axis_id, find the best (lowest) priority it matched
w_best_priority = Window.partitionBy("axis_id")
brd_matches = (
    candidates_layer1
    .withColumn("_best_priority", F.min("priority").over(w_best_priority))
    .filter(F.col("priority") == F.col("_best_priority"))
    .drop("_best_priority")
)

# Add match layer label + auditability columns
brd_matches = (
    brd_matches
    .withColumn("MatchLayer", F.lit("BRD"))
    .withColumn("run_id", F.lit(RUN_ID))
    .withColumn("batch_id", F.lit(BATCH_ID))
    .withColumn("rule_version", F.lit(RULE_VERSION))
    .withColumn("match_timestamp", F.lit(RUN_TIMESTAMP))
)

# Persist — materialises candidates in one pipeline
brd_matches = brd_matches.persist(StorageLevel.MEMORY_AND_DISK)

# ⚡ Single .count() here — materialises everything.
brd_match_rows = brd_matches.count()

# Distinct axis and finstore counts (Pandas equivalent: .nunique())
brd_unique_axis = brd_matches.select("axis_id").distinct().count()
brd_unique_fin  = brd_matches.select("fin_id").distinct().count()

brd_match_rate = brd_unique_axis / ORIGINAL_AXIS_COUNT * 100

print(f"\n{'='*80}")
print("LAYER 1 (BRD) RESULTS  —  Waterfall Parity")
print(f"{'='*80}")
print(f"Total match rows:    {brd_match_rows:,}  (within-rule many-to-many)")
print(f"Unique Axis matched: {brd_unique_axis:,}  (each axis → best rule only)")
print(f"Unique Finstore used:{brd_unique_fin:,}")
print(f"BRD Match Rate (unique axis): {brd_match_rate:.2f}%")
print(f"Remaining unmatched: {ORIGINAL_AXIS_COUNT - brd_unique_axis:,}")

# ⚡ Breakdown by rule: reads from cached brd_matches
print("\nMatches by rule:")
brd_matches.groupBy("priority", "category", "description").count() \
    .orderBy("priority").show(20, truncate=False)

---
### 🔹 Section 11b: Layer 2 — Greedy / Probabilistic Matching

**Key Spark optimisations vs. Pandas:**
- No Python `iterrows()` loops — everything is join-based
- Strategy 1: blocked join on `counterpartyid` + tolerance filter
- Strategy 2: blocked join on amount buckets (±1 bucket) + strict tolerance filter
- `resolve_matches()` window dedup: picks **1 best fin per axis** — matches
  the Pandas `nsmallest(1, "amount_diff")` logic in the greedy loop

---
### 🔹 Section 11: Compute Unmatched Pools for Greedy

In [ ]:
# ============================================================
# COMPUTE UNMATCHED POOLS
# ============================================================
# ⚡ PERF: No .count() — pools are lazy. They materialise when
#    greedy strategies read them (via their own cache).
#
# brd_matches may have within-rule many-to-many, so we use DISTINCT
# axis_id / fin_id for pool removal — identical to Pandas
# set(matched[idx].unique()).

# Distinct Axis / Fin IDs matched in Layer 1
matched_axis_ids = brd_matches.select("axis_id").distinct()
matched_fin_ids  = brd_matches.select("fin_id").distinct()

# Anti-join to get unmatched
axis_unmatched = axis_core.join(matched_axis_ids, on="axis_id", how="left_anti")
fin_unmatched = fin_core.join(matched_fin_ids, on="fin_id", how="left_anti")

# Ensure amounts are clean
axis_unmatched = axis_unmatched.filter(
    F.col(AXIS_AMOUNT_COL).isNotNull() & (F.col(AXIS_AMOUNT_COL) != 0)
)
fin_unmatched = fin_unmatched.filter(
    F.col(FIN_AMOUNT_COL).isNotNull()
)

# Normalise counterparty for join
axis_unmatched = axis_unmatched.withColumn(
    "cpty_str", F.trim(F.col("CounterpartyId").cast("string"))
)
fin_unmatched = fin_unmatched.withColumn(
    "cpty_str", F.trim(F.col("counterpartyid").cast("string"))
)

# Persist — materialises during greedy Strategy 1
axis_unmatched = axis_unmatched.persist(StorageLevel.MEMORY_AND_DISK)
fin_unmatched = fin_unmatched.persist(StorageLevel.MEMORY_AND_DISK)

print(f"\n{'='*80}")
print("LAYER 2: GREEDY / PROBABILISTIC MATCHING")
print(f"{'='*80}")
print("Unmatched pools prepared (will materialise during Strategy 1).")

---
### 🔹 Section 12: Greedy Strategy 1 — Amount + Counterparty (1%)

Replaces the Pandas `groupby → iterrows` loop with a **join on counterparty + tolerance filter**.

In [ ]:
# ============================================================
# GREEDY STRATEGY 1: Amount + Counterparty (1% tolerance)
# ============================================================
# Blocked join on counterparty → filter by tolerance → rank best match.

print(f"\n--- Strategy 1: Amount + Counterparty ({GREEDY_TOLERANCE_PCT*100}% tolerance) ---")

# Join on counterparty (blocking key)
greedy1_candidates = (
    axis_unmatched.alias("a")
    .join(
        fin_unmatched.alias("f"),
        on=(F.col("a.cpty_str") == F.col("f.cpty_str")),
        how="inner"
    )
    # Filter: exclude nan/empty counterparties
    .filter(
        (F.col("a.cpty_str").isNotNull()) &
        (F.col("a.cpty_str") != "") &
        (F.col("a.cpty_str") != "nan")
    )
    # Compute difference and tolerance
    .withColumn("amount_diff", F.abs(F.col(f"a.{AXIS_AMOUNT_COL}") - F.col(f"f.{FIN_AMOUNT_COL}")))
    .withColumn("tolerance", F.abs(F.col(f"a.{AXIS_AMOUNT_COL}")) * GREEDY_TOLERANCE_PCT)
    # Filter within tolerance
    .filter(F.col("amount_diff") <= F.col("tolerance"))
    # Select candidate edge columns
    .select(
        F.col("a.axis_id").alias("axis_id"),
        F.col("f.fin_id").alias("fin_id"),
        F.lit(16).cast("int").alias("priority"),
        F.lit("GREEDY").alias("category"),
        F.lit(1).cast("int").alias("brd_priority"),
        F.lit(f"Greedy: Amount+Counterparty ({GREEDY_TOLERANCE_PCT*100}%)").alias("description"),
        F.col("amount_diff"),
        (F.col("amount_diff") / F.greatest(F.abs(F.col(f"a.{AXIS_AMOUNT_COL}")), F.lit(1e-9))).alias("amount_rel_diff"),
        F.lit(0).cast("int").alias("key_strength"),
    )
)

# Resolve best-fin-per-axis (many-to-one allowed)
greedy1_matches = resolve_matches(greedy1_candidates)
greedy1_matches = (
    greedy1_matches
    .withColumn("MatchLayer", F.lit("GREEDY"))
    .withColumn("run_id", F.lit(RUN_ID))
    .withColumn("batch_id", F.lit(BATCH_ID))
    .withColumn("rule_version", F.lit(RULE_VERSION))
    .withColumn("match_timestamp", F.lit(RUN_TIMESTAMP))
)
greedy1_matches = greedy1_matches.persist(StorageLevel.MEMORY_AND_DISK)

# ⚡ This .count() is needed — Strategy 2 uses anti-join on these IDs
greedy1_count = greedy1_matches.count()
print(f"Strategy 1 matches: {greedy1_count:,}")

---
### 🔹 Section 13: Greedy Strategy 2 — Amount Only (0.1%)

Uses **amount bucket blocking** to avoid full cross-join.
Each Axis record expands to 3 bucket rows (bucket-1, bucket, bucket+1)
then joins Finstore on bucket.

In [ ]:
# ============================================================
# GREEDY STRATEGY 2: Amount Only (0.1% tolerance) with Bucket Blocking
# ============================================================

print(f"\n--- Strategy 2: Amount only ({STRICT_TOLERANCE_PCT*100}% tolerance) ---")

# Remove records already matched in Strategy 1
greedy1_axis_ids = greedy1_matches.select("axis_id")
greedy1_fin_ids = greedy1_matches.select("fin_id")

axis_remaining_s2 = axis_unmatched.join(greedy1_axis_ids, on="axis_id", how="left_anti")
fin_remaining_s2 = fin_unmatched.join(greedy1_fin_ids, on="fin_id", how="left_anti")

# ⚡ PERF: Removed .count() that was here — it was purely diagnostic
#    and triggered an extra Spark job on the remaining pool.

# Create amount buckets (native Spark — no UDF)
axis_remaining_s2 = axis_remaining_s2.withColumn(
    "amount_bucket",
    (F.floor(F.col(AXIS_AMOUNT_COL) / BUCKET_SIZE) * BUCKET_SIZE).cast("long")
)

fin_remaining_s2 = fin_remaining_s2.withColumn(
    "amount_bucket",
    (F.floor(F.col(FIN_AMOUNT_COL) / BUCKET_SIZE) * BUCKET_SIZE).cast("long")
)

# Expand axis buckets to ±1 for neighbour search
# This creates 3 rows per axis trade — much cheaper than cross-join
bucket_offsets = spark.createDataFrame(
    [(-BUCKET_SIZE,), (0,), (BUCKET_SIZE,)],
    ["_offset"]
)

axis_expanded = (
    axis_remaining_s2
    .crossJoin(F.broadcast(bucket_offsets))
    .withColumn("search_bucket", F.col("amount_bucket") + F.col("_offset"))
    .drop("_offset")
)

# Join on bucket
greedy2_candidates = (
    axis_expanded.alias("a")
    .join(
        fin_remaining_s2.alias("f"),
        on=(F.col("a.search_bucket") == F.col("f.amount_bucket")),
        how="inner"
    )
    .withColumn("amount_diff", F.abs(F.col(f"a.{AXIS_AMOUNT_COL}") - F.col(f"f.{FIN_AMOUNT_COL}")))
    .withColumn("tolerance", F.abs(F.col(f"a.{AXIS_AMOUNT_COL}")) * STRICT_TOLERANCE_PCT)
    .filter(F.col("amount_diff") <= F.col("tolerance"))
    .dropDuplicates(["axis_id", "fin_id"])
    .select(
        F.col("a.axis_id").alias("axis_id"),
        F.col("f.fin_id").alias("fin_id"),
        F.lit(17).cast("int").alias("priority"),
        F.lit("GREEDY").alias("category"),
        F.lit(2).cast("int").alias("brd_priority"),
        F.lit(f"Greedy: Amount Strict ({STRICT_TOLERANCE_PCT*100}%)").alias("description"),
        F.col("amount_diff"),
        (F.col("amount_diff") / F.greatest(F.abs(F.col(f"a.{AXIS_AMOUNT_COL}")), F.lit(1e-9))).alias("amount_rel_diff"),
        F.lit(0).cast("int").alias("key_strength"),
    )
)

# Resolve best-fin-per-axis (many-to-one allowed)
greedy2_matches = resolve_matches(greedy2_candidates)
greedy2_matches = (
    greedy2_matches
    .withColumn("MatchLayer", F.lit("GREEDY"))
    .withColumn("run_id", F.lit(RUN_ID))
    .withColumn("batch_id", F.lit(BATCH_ID))
    .withColumn("rule_version", F.lit(RULE_VERSION))
    .withColumn("match_timestamp", F.lit(RUN_TIMESTAMP))
)
greedy2_matches = greedy2_matches.persist(StorageLevel.MEMORY_AND_DISK)

greedy2_count = greedy2_matches.count()
print(f"Strategy 2 matches: {greedy2_count:,}")

---
### 🔹 Section 14: Greedy Layer Summary

In [ ]:
# ============================================================
# LAYER 2 (GREEDY) SUMMARY
# ============================================================
# ⚡ No new Spark jobs — uses pre-computed counts.

total_greedy = greedy1_count + greedy2_count

print(f"\n{'='*60}")
print("LAYER 2 (GREEDY) SUMMARY")
print(f"{'='*60}")
print(f"Strategy 1 (Amount+Counterparty): {greedy1_count:,}")
print(f"Strategy 2 (Amount Strict):       {greedy2_count:,}")
print(f"Total Greedy matches:             {total_greedy:,}")

---
## 🔹 Section 15: Final Results Consolidation

In [ ]:
# ============================================================
# FINAL CONSOLIDATION  —  Waterfall Parity
# ============================================================
# BRD layer: best-rule per axis, all fin matches within that rule
#   (1 axis can appear N times if its best rule hit N finstores)
# Greedy layers: 1 row per axis (resolve_matches window dedup)
#
# NO dropDuplicates on axis_id — within-rule many-to-many is intentional.
# Cross-rule duplication is already prevented by the best-priority filter.
#
# Pool removal already used DISTINCT axis_id / fin_id sets so
# greedy layers only process truly unmatched trades.

# Union all match layers (lazy)
all_matches = (
    brd_matches
    .unionByName(greedy1_matches)
    .unionByName(greedy2_matches)
    # NO dropDuplicates — within-rule many-to-many rows are intentional
)
all_matches = all_matches.persist(StorageLevel.MEMORY_AND_DISK)

# ── Count total rows and distinct axis IDs ───────────────────────────────
total_match_rows = all_matches.count()
total_unique_axis = all_matches.select("axis_id").distinct().count()
total_unique_fin  = all_matches.select("fin_id").distinct().count()

# Greedy counts (already 1 row per axis from resolve_matches)
total_greedy = greedy1_count + greedy2_count

# Unique axis matched = BRD unique + greedy (greedy axes are guaranteed
# disjoint from BRD axes thanks to anti-join pool removal)
total_matched = brd_unique_axis + total_greedy
final_match_rate = total_matched / ORIGINAL_AXIS_COUNT * 100

# Unmatched axis count
unmatched_axis_count = ORIGINAL_AXIS_COUNT - total_matched

# ── Cross-layer axis uniqueness assertion (greedy vs BRD) ────────────────
# Greedy axes must NOT overlap with BRD axes (pool anti-join ensures this).
greedy_axis = greedy1_matches.select("axis_id").union(greedy2_matches.select("axis_id"))
overlap = brd_matches.select("axis_id").intersect(greedy_axis)
overlap_count = overlap.count()

if overlap_count > 0:
    print(f"⚠️  WARNING: {overlap_count:,} axis_id(s) matched in BOTH BRD and Greedy layers!")
    print("    Investigate pool anti-join logic.")
else:
    print("✅ Cross-layer axis uniqueness check PASSED — no axis_id matched in both BRD and Greedy.")

# ── Verify total_unique_axis matches expected count ──────────────────────
if total_unique_axis != total_matched:
    print(f"⚠️  WARNING: distinct axis in all_matches ({total_unique_axis:,}) != "
          f"expected total_matched ({total_matched:,})")
    total_matched = total_unique_axis  # use actual safe count
else:
    print(f"✅ Unique axis count verified: {total_unique_axis:,}")

# Summary
many_to_many_rows = total_match_rows - total_unique_axis
print(f"ℹ️  Within-rule many-to-many: {many_to_many_rows:,} extra rows "
      f"({total_match_rows:,} total rows, {total_unique_axis:,} unique axis, "
      f"{total_unique_fin:,} unique fin)")

# Build unmatched sets using DISTINCT axis/fin IDs
all_matched_axis_ids = all_matches.select("axis_id").distinct()
all_matched_fin_ids  = all_matches.select("fin_id").distinct()

final_unmatched_axis = axis_core.join(all_matched_axis_ids, on="axis_id", how="left_anti")
final_unmatched_fin  = fin_core.join(all_matched_fin_ids, on="fin_id", how="left_anti")

print(f"\n{'='*80}")
print("FINAL RESULTS")
print(f"{'='*80}")
print(f"SCOPE: {'SOPHIS/DELTA1 EXCLUDED' if EXCLUDE_SOPHIS_DELTA1 else 'ALL SYSTEMS'}")
print(f"{'-'*60}")
print(f"Total Axis (in-scope): {ORIGINAL_AXIS_COUNT:,}")
print(f"\nLayer 1 (BRD Rules):")
print(f"  {brd_unique_axis:,} unique axis matched, {brd_match_rows:,} match rows "
      f"(within-rule m:m) ({brd_match_rate:.2f}%)")
print(f"Layer 2 (Greedy):")
print(f"  {total_greedy:,} matched ({total_greedy/ORIGINAL_AXIS_COUNT*100:.2f}%)")
print(f"\nTOTAL UNIQUE AXIS MATCHED: {total_matched:,}")
print(f"TOTAL MATCH ROWS:          {total_match_rows:,}")
print(f"TOTAL UNMATCHED (Axis):    {ORIGINAL_AXIS_COUNT - total_matched:,}")
print(f"\n>>> FINAL MATCH RATE (unique axis): {total_matched / ORIGINAL_AXIS_COUNT * 100:.2f}% <<<")
print(f"{'='*80}")

---
### 🔹 Section 15a: Match Count Verification

Three assertions that **must** all pass before saving any results:
1. No `axis_id` appears in both BRD-matched and Greedy-matched sets (pool isolation)
2. No `axis_id` appears in both matched and unmatched sets
3. Unique matched axis + unmatched axis = total in-scope Axis count

> **Note**: `axis_id` **can** appear multiple times within BRD matches
> if its best rule produced multiple finstore hits (within-rule many-to-many).
> The checks above verify **unique axis** counts, not row counts.

In [ ]:
# ============================================================
# SECTION 15a: MATCH COUNT VERIFICATION
# ============================================================
# These checks ensure the numbers in every downstream report are
# correct.  If any assertion fails the notebook should STOP.
#
# NOTE: all_matches has within-rule many-to-many rows in the BRD layer,
# so we verify UNIQUE AXIS counts, not raw row counts.

print(f"\n{'='*80}")
print("MATCH COUNT VERIFICATION")
print(f"{'='*80}")

errors = []

# ── Check 1: no axis_id in both BRD and Greedy (pool isolation) ──────
greedy_distinct_axis = (
    greedy1_matches.select("axis_id")
    .union(greedy2_matches.select("axis_id"))
    .distinct()
)
brd_greedy_overlap = brd_matches.select("axis_id").distinct().intersect(greedy_distinct_axis)
brd_greedy_overlap_count = brd_greedy_overlap.count()
if brd_greedy_overlap_count != 0:
    errors.append(
        f"CHECK 1 FAILED: {brd_greedy_overlap_count:,} axis_id(s) matched "
        f"in BOTH BRD and Greedy layers — pool anti-join is broken."
    )
else:
    print(f"✅ CHECK 1 PASSED: zero axis_id overlap between BRD and Greedy layers")

# ── Check 2: no axis_id in both matched and unmatched ────────────────
overlap_axis = all_matches.select("axis_id").distinct().intersect(final_unmatched_axis.select("axis_id"))
overlap_count = overlap_axis.count()
if overlap_count != 0:
    errors.append(
        f"CHECK 2 FAILED: {overlap_count:,} axis_id(s) appear in BOTH "
        f"matched and unmatched sets"
    )
else:
    print(f"✅ CHECK 2 PASSED: zero axis_id overlap between matched and unmatched")

# ── Check 3: unique matched axis + unmatched == ORIGINAL_AXIS_COUNT ──
unmatched_axis_actual = final_unmatched_axis.count()
total_accounted = total_matched + unmatched_axis_actual
if total_accounted != ORIGINAL_AXIS_COUNT:
    errors.append(
        f"CHECK 3 FAILED: matched_unique_axis({total_matched:,}) + "
        f"unmatched({unmatched_axis_actual:,}) = {total_accounted:,} "
        f"!= ORIGINAL_AXIS_COUNT({ORIGINAL_AXIS_COUNT:,})"
    )
else:
    print(f"✅ CHECK 3 PASSED: matched_unique_axis({total_matched:,}) + "
          f"unmatched({unmatched_axis_actual:,}) == "
          f"ORIGINAL_AXIS_COUNT({ORIGINAL_AXIS_COUNT:,})")

# ── Fail loudly if any check failed ─────────────────────────────────
if errors:
    print(f"\n{'!'*80}")
    for e in errors:
        print(f"  ❌ {e}")
    print(f"{'!'*80}")
    raise AssertionError(
        f"{len(errors)} verification check(s) failed — "
        f"do NOT trust downstream reports until resolved."
    )
else:
    print(f"\n{'='*80}")
    print("ALL VERIFICATION CHECKS PASSED ✅")
    print(f"{'='*80}")

---
## 🔹 Section 15b: Save Base Matches (Narrow — Before Enrichment)

Save the core match results **before** the expensive wide-column enrichment join.
This provides a lightweight Delta table (~15 columns) for quick analysis,
while the enriched version with 100+ columns is saved separately after.

In [ ]:
# ============================================================
# SAVE BASE MATCHES — narrow schema, before enrichment
# ============================================================
# This is the lightweight output (~15 cols) for quick downstream queries.
# The enriched wide table (100+ cols) is saved separately in Section 19.

print(f"Saving base (narrow) match results to: {OUTPUT_DIR}")
print(f"{'-'*60}")

# --- All matches (BRD + Greedy) — narrow ---
all_matches.write.format("delta").mode("overwrite").save(f"{OUTPUT_DIR}/matched_all_base")
print(f"✅ Saved Delta: matched_all_base ({total_match_rows:,} rows, {len(all_matches.columns)} cols)")

# --- BRD only (narrow) ---
brd_matches.write.format("delta").mode("overwrite").save(f"{OUTPUT_DIR}/matched_brd_layer")
print(f"✅ Saved Delta: matched_brd_layer ({brd_match_rows:,} rows)")

# --- Greedy only (narrow) ---
# ⚡ CORRECTNESS: reuse all_matches (already deduplication-guarded) instead of
#    re-unioning greedy1 + greedy2 — avoids redundant lineage and guarantees
#    consistency with the dedup guard applied in Section 15.
greedy_all_df = all_matches.filter(F.col("MatchLayer") == "GREEDY")
greedy_all_df.write.format("delta").mode("overwrite").save(f"{OUTPUT_DIR}/matched_greedy_layer")
print(f"✅ Saved Delta: matched_greedy_layer ({total_greedy:,} rows)")

# --- Unmatched Axis (narrow core only) ---
final_unmatched_axis.write.format("delta").mode("overwrite").save(f"{OUTPUT_DIR}/unmatched_axis_base")
print(f"✅ Saved Delta: unmatched_axis_base")

# --- Unmatched Finstore (narrow core only) ---
final_unmatched_fin.write.format("delta").mode("overwrite").save(f"{OUTPUT_DIR}/unmatched_finstore_base")
print(f"✅ Saved Delta: unmatched_finstore_base")

print(f"\n{'='*60}")
print("BASE (NARROW) RESULTS SAVED — enrichment follows in Section 16.")
print(f"{'='*60}")

---
## 🔹 Section 16: Enrichment — Join Back Wide Columns

Now that matching is done on narrow core tables,
we join back the full 100+ columns for the final output.

In [ ]:
# ============================================================
# ENRICHMENT — Join back wide columns (100+ cols)
# ============================================================
# This is the ONLY place where wide DataFrames are joined.
# All matching was done on narrow core tables.
# ⚡ PERF: No .count() — these are written directly to Delta.
#    Counts come from the narrow base already computed.

# Rename wide columns to avoid collisions
axis_wide_renamed = axis_wide
for c in axis_wide.columns:
    if c != "axis_id":
        axis_wide_renamed = axis_wide_renamed.withColumnRenamed(c, f"{c}_Axis")

fin_wide_renamed = fin_wide
for c in fin_wide.columns:
    if c != "fin_id":
        fin_wide_renamed = fin_wide_renamed.withColumnRenamed(c, f"{c}_Finstore")

# Enrich matches
matches_enriched = (
    all_matches
    .join(axis_wide_renamed, on="axis_id", how="left")
    .join(fin_wide_renamed, on="fin_id", how="left")
)

# Add Variance column
axis_amt_col_enriched = f"{AXIS_AMOUNT_COL}_Axis"
fin_amt_col_enriched = f"{FIN_AMOUNT_COL}_Finstore"

matches_enriched = matches_enriched.withColumn(
    "Variance",
    F.col(axis_amt_col_enriched) - F.col(fin_amt_col_enriched)
)

# Enrich unmatched axis
unmatched_axis_enriched = (
    final_unmatched_axis
    .select("axis_id")
    .join(axis_wide_renamed, on="axis_id", how="left")
)

# Enrich unmatched finstore
unmatched_fin_enriched = (
    final_unmatched_fin
    .select("fin_id")
    .join(fin_wide_renamed, on="fin_id", how="left")
)

# ⚡ No .count() here — enriched DFs are written directly to Delta in Section 19.
print(f"Enriched DataFrames prepared (lazy).")
print(f"  matches_enriched columns: {len(matches_enriched.columns)}")
print(f"  unmatched_axis_enriched columns: {len(unmatched_axis_enriched.columns)}")
print(f"  unmatched_fin_enriched columns: {len(unmatched_fin_enriched.columns)}")
print("Will materialise during Delta write in Section 19.")

---
## 🔹 Section 17: Matches by System Breakdown

In [ ]:
# ============================================================
# BREAKDOWN: Matches by Source System
# ============================================================

print(f"\n{'='*60}")
print("MATCHES BY SOURCE SYSTEM")
print(f"{'='*60}")

# BRD matches by system
print("\nBRD Matches by system:")
brd_by_system = (
    brd_matches
    .join(axis_core.select("axis_id", "SourceSystemName"), on="axis_id", how="left")
    .groupBy("SourceSystemName")
    .count()
    .orderBy(F.desc("count"))
)
brd_by_system.show(20, truncate=False)

# Greedy matches by system
print("Greedy Matches by system:")
greedy_all = greedy1_matches.unionByName(greedy2_matches)
greedy_by_system = (
    greedy_all
    .join(axis_core.select("axis_id", "SourceSystemName"), on="axis_id", how="left")
    .groupBy("SourceSystemName")
    .count()
    .orderBy(F.desc("count"))
)
greedy_by_system.show(20, truncate=False)

---
## 🔹 Section 18: Remaining Unmatched by System

In [ ]:
# ============================================================
# UNMATCHED BY SYSTEM
# ============================================================

print(f"\n{'='*60}")
print("REMAINING UNMATCHED BY SYSTEM")
print(f"{'='*60}")

unmatched_by_system = (
    final_unmatched_axis
    .groupBy("SourceSystemName")
    .count()
    .orderBy(F.desc("count"))
)
unmatched_by_system.show(20, truncate=False)

---
## 🔹 Section 18b: Data Quality Validation (Best Practice §7)

Validate core column data contracts before outputs are consumed downstream.

In [ ]:
# ============================================================
# DATA QUALITY VALIDATION  (Best Practice §7)
# ============================================================
# ⚡ PERF: Instead of N separate .count() calls (was 14 Spark jobs!),
#    compute all violation counts in a SINGLE aggregation per DataFrame.

def validate_dataframe_fast(df, name, checks):
    """Run all checks in a single aggregation pass — 1 Spark job per DF."""
    agg_exprs = [
        F.sum(F.when(expr, 1).otherwise(0)).alias(desc)
        for desc, expr in checks
    ]
    result_row = df.agg(*agg_exprs).collect()[0]
    rows = [(name, desc, int(result_row[desc])) for desc, _ in checks]
    return spark.createDataFrame(rows, ["dataset", "check", "violation_count"])

# --- Axis Core checks (1 Spark job instead of 5) ---
axis_checks = [
    ("null_axis_id",            F.col("axis_id").isNull()),
    ("null_CounterpartyId",     F.col("CounterpartyId").isNull()),
    ("null_NotionalAmount",     F.col(AXIS_AMOUNT_COL).isNull()),
    ("negative_Notional",       F.col(AXIS_AMOUNT_COL) < 0),
    ("null_ReconSubProduct",    F.col("ReconSubProduct").isNull()),
]

# --- Finstore Core checks (1 Spark job instead of 4) ---
fin_checks = [
    ("null_fin_id",             F.col("fin_id").isNull()),
    ("null_counterpartyid",     F.col("counterpartyid").isNull()),
    ("null_NotionalAmount",     F.col(FIN_AMOUNT_COL).isNull()),
    ("negative_Notional",       F.col(FIN_AMOUNT_COL) < 0),
]

# --- Match output checks (1 Spark job instead of 5) ---
match_checks = [
    ("null_axis_id",        F.col("axis_id").isNull()),
    ("null_fin_id",         F.col("fin_id").isNull()),
    ("null_description",    F.col("description").isNull()),
    ("null_run_id",         F.col("run_id").isNull()),
    ("amount_rel_diff_gt_1", F.col("amount_rel_diff") > 1.0),
]

dq_axis    = validate_dataframe_fast(axis_core,   "axis_core",   axis_checks)
dq_fin     = validate_dataframe_fast(fin_core,    "fin_core",    fin_checks)
dq_matches = validate_dataframe_fast(all_matches, "all_matches", match_checks)

dq_report = dq_axis.union(dq_fin).union(dq_matches)
print("=== Data Quality Validation (3 Spark jobs instead of 14) ===")
dq_report.show(50, truncate=False)

violations = dq_report.filter(F.col("violation_count") > 0).count()
if violations > 0:
    print(f"⚠️  {violations} check(s) have violations — review before promoting to Gold.")
else:
    print("✅ All data quality checks passed.")

---
## 🔹 Section 18c: Explainability — Unmatched Reason Breakdown (Best Practice §6)

Every unmatched trade should carry a *reason* so downstream consumers can investigate.

In [ ]:
# ============================================================
# EXPLAINABILITY — UNMATCHED REASON BREAKDOWN  (Best Practice §6)
# Classify why each unmatched trade failed to match.
# ============================================================

# --- Axis unmatched reasons ---
# Check if the trade appeared as a candidate in ANY rule but was outcompeted
axis_candidates_ever = candidates_layer1.select("axis_id").distinct()

axis_unmatched_reasons = (
    final_unmatched_axis
    .join(axis_candidates_ever, on="axis_id", how="left")
    .withColumn(
        "unmatched_reason",
        F.when(axis_candidates_ever["axis_id"].isNotNull(),
               F.lit("candidate_existed_but_consumed_by_higher_priority"))
         .otherwise(F.lit("no_candidate_key_found"))
    )
    .select("axis_id", "SourceSystemTradeId", "CounterpartyId", "ReconSubProduct",
            AXIS_AMOUNT_COL, "unmatched_reason")
    .withColumn("run_id", F.lit(RUN_ID))
    .withColumn("batch_id", F.lit(BATCH_ID))
)

# --- Finstore unmatched reasons ---
fin_candidates_ever = candidates_layer1.select("fin_id").distinct()

fin_unmatched_reasons = (
    final_unmatched_fin
    .join(fin_candidates_ever, on="fin_id", how="left")
    .withColumn(
        "unmatched_reason",
        F.when(fin_candidates_ever["fin_id"].isNotNull(),
               F.lit("candidate_existed_but_consumed_by_higher_priority"))
         .otherwise(F.lit("no_candidate_key_found"))
    )
    .select("fin_id", "tradeid", "counterpartyid",
            FIN_AMOUNT_COL, "unmatched_reason")
    .withColumn("run_id", F.lit(RUN_ID))
    .withColumn("batch_id", F.lit(BATCH_ID))
)

# --- Summary ---
print("=== Unmatched Axis Reason Breakdown ===")
axis_unmatched_reasons.groupBy("unmatched_reason").count().show(truncate=False)

print("=== Unmatched Finstore Reason Breakdown ===")
fin_unmatched_reasons.groupBy("unmatched_reason").count().show(truncate=False)

# Optionally persist for audit
# axis_unmatched_reasons.write.format("delta").mode("overwrite").save(f"{OUTPUT_DIR}/unmatched_axis_reasons")
# fin_unmatched_reasons.write.format("delta").mode("overwrite").save(f"{OUTPUT_DIR}/unmatched_fin_reasons")

---
## 🔹 Section 19: Save Results

**Best practice:** Save as Delta for downstream consumption;
export CSV only for legacy consumers / samples.

In [ ]:
# ============================================================
# SAVE ENRICHED RESULTS — Delta (wide tables with 100+ columns)
# ============================================================
# Base (narrow) results were already saved in Section 15b.
# This section saves the enriched (wide) versions for full reporting.
# ⚡ PERF: No .count() — write directly. Counts use previously computed values.

print(f"Saving enriched (wide) results to: {OUTPUT_DIR}")
print(f"{'-'*60}")

# All matches enriched (wide — full 100+ columns)
matches_enriched.write.format("delta").mode("overwrite").save(f"{OUTPUT_DIR}/matched_all_enriched")
print(f"✅ Saved Delta: matched_all_enriched ({total_matched:,} rows)")

# Unmatched Axis (wide)
unmatched_axis_enriched.write.format("delta").mode("overwrite").save(f"{OUTPUT_DIR}/unmatched_axis_enriched")
print(f"✅ Saved Delta: unmatched_axis_enriched")

# Unmatched Finstore (wide)
unmatched_fin_enriched.write.format("delta").mode("overwrite").save(f"{OUTPUT_DIR}/unmatched_finstore_enriched")
print(f"✅ Saved Delta: unmatched_finstore_enriched")

# ---- OPTIMIZE + ZORDER (Best Practice §5 — Partitioning & Layout) ----
# With delta.optimizeWrite + delta.autoCompact enabled in Section 1,
# files are already well-sized. ZORDER further optimises read patterns.
# Uncomment on Databricks:
#
# spark.sql(f"OPTIMIZE delta.`{OUTPUT_DIR}/matched_all_enriched` ZORDER BY (axis_id, fin_id, priority)")
# spark.sql(f"OPTIMIZE delta.`{OUTPUT_DIR}/matched_all_base` ZORDER BY (axis_id, priority)")
# spark.sql(f"OPTIMIZE delta.`{OUTPUT_DIR}/unmatched_axis_enriched` ZORDER BY (axis_id)")
# spark.sql(f"OPTIMIZE delta.`{OUTPUT_DIR}/unmatched_finstore_enriched` ZORDER BY (fin_id)")
# print("Delta tables OPTIMIZED + ZORDERed.")

print("\nAll enriched results saved successfully.")

---
## 🔹 Section 20: Summary Report

In [ ]:
# ============================================================
# SUMMARY REPORT
# ============================================================
# ⚡ PERF: Uses only pre-computed counts — no new Spark jobs.
#    Report saved via dbutils (Databricks) or Python file I/O,
#    NOT via spark.sparkContext.parallelize (which triggers a job).

report = f"""
{'='*80}
DERIVATIVES MATCHING — COMPLETE RESULTS (BRD + GREEDY)
PySpark / Databricks Production
{'='*80}

Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
Run ID:    {RUN_ID}
Batch ID:  {BATCH_ID}
Rule Ver:  {RULE_VERSION}
Scope:     {'SOPHIS/DELTA1 EXCLUDED' if EXCLUDE_SOPHIS_DELTA1 else 'ALL SYSTEMS'}

{'='*80}
INPUT DATA
{'='*80}
Total Axis (original):  {ORIGINAL_AXIS_COUNT_FULL:,}
Total Finstore:         {ORIGINAL_FINSTORE_COUNT:,}
Excluded (SOPHIS/DELTA1): {excluded_count:,}
Axis (in-scope):        {ORIGINAL_AXIS_COUNT:,}

{'='*80}
LAYER 1: BRD DETERMINISTIC MATCHING
{'='*80}
Rules executed: 15 (3 SOPHIS + 10 OTC + 2 ETD)
BRD match rows (m:m): {brd_match_rows:,}
Unique Axis matched: {brd_unique_axis:,}
Unique Finstore used: {brd_unique_fin:,}
Match Rate (unique axis): {brd_match_rate:.2f}%

{'='*80}
LAYER 2: GREEDY/PROBABILISTIC MATCHING
{'='*80}
Strategy 1 (Amount+Counterparty, {GREEDY_TOLERANCE_PCT*100:.1f}%): {greedy1_count:,}
Strategy 2 (Amount Strict, {STRICT_TOLERANCE_PCT*100:.1f}%): {greedy2_count:,}
Total Greedy: {total_greedy:,}

{'='*80}
COMBINED RESULTS
{'='*80}
Total Unique Axis Matched: {total_matched:,}
Total Match Rows:    {total_match_rows:,}
Total Unmatched:     {unmatched_axis_count:,}
Final Match Rate (unique axis): {final_match_rate:.2f}%

{'='*80}
OUTPUT TABLES
{'='*80}
- matched_all_base       (narrow — match metadata, {total_match_rows:,} rows)
- matched_brd_layer      (narrow — BRD only, {brd_match_rows:,} rows)
- matched_greedy_layer   (narrow — Greedy only, {total_greedy:,} rows)
- matched_all_enriched   (wide — 100+ cols, {total_match_rows:,} rows)
- unmatched_axis_base    (narrow — core cols)
- unmatched_axis_enriched(wide — all cols)
- unmatched_finstore_base(narrow — core cols)
- unmatched_finstore_enriched (wide — all cols)

{'='*80}
SCORING & AUDITABILITY (Best Practice §2D, §6)
{'='*80}
- Every match row includes: amount_diff, amount_rel_diff, key_strength
- Auditability columns: run_id, batch_id, rule_version, match_timestamp
- Unmatched reason breakdown: see Explainability section above

{'='*80}
PERFORMANCE OPTIMISATIONS APPLIED
{'='*80}
- Core/Wide split: matching on {len(AXIS_CORE_COLS)} Axis + {len(FIN_CORE_COLS)} Finstore columns
- Candidate edges: BRD keeps best-rule per axis (waterfall parity), Greedy uses window ranking
- Blocked joins for greedy (counterparty / amount buckets)
- Native Spark SQL functions (no Python UDFs)
- AQE + skew join + local shuffle reader enabled
- Shuffle partitions: 320 (tuned for 160-core cluster)
- Broadcast threshold: 256 MB (auto-broadcasts small join sides)
- Delta optimizeWrite + autoCompact enabled
- MEMORY_AND_DISK persist for scale safety
- Eliminated ~15 unnecessary .count() Spark jobs

{'='*80}
DATA QUALITY (Best Practice §7)
{'='*80}
- Core column null/range checks executed (single-pass aggregation)
{'='*80}
"""

print(report)

# Save report — use dbutils on Databricks, or Python file I/O locally
try:
    # Databricks
    dbutils.fs.put(f"{OUTPUT_DIR}/matching_summary_report.txt", report, overwrite=True)
    print(f"Summary report saved via dbutils to: {OUTPUT_DIR}/matching_summary_report.txt")
except NameError:
    # Local / non-Databricks
    import os
    report_path = OUTPUT_DIR.replace("dbfs:", "/dbfs") if OUTPUT_DIR.startswith("dbfs:") else OUTPUT_DIR
    os.makedirs(report_path, exist_ok=True)
    with open(os.path.join(report_path, "matching_summary_report.txt"), "w") as f:
        f.write(report)
    print(f"Summary report saved locally to: {report_path}/matching_summary_report.txt")

---
## 🔹 Section 21: Cleanup — Unpersist Cached DataFrames

In [ ]:
# ============================================================
# CLEANUP — release persisted DataFrames
# ============================================================

for df_to_unpersist in [axis_core, fin_core, candidates_layer1,
                         brd_matches, axis_unmatched, fin_unmatched,
                         greedy1_matches, greedy2_matches, all_matches]:
    try:
        df_to_unpersist.unpersist()
    except Exception:
        pass

print("All persisted DataFrames unpersisted. Pipeline complete.")

---
## 🔹 Section 22: Accuracy Report

Detailed breakdown of match results across all layers, rules, and strategies.
Mirrors the summary report from the Pandas reference notebook so results
can be compared directly between implementations.

In [ ]:
# ============================================================
# SECTION 22: ACCURACY REPORT
# ============================================================
# Mirrors the Pandas notebook summary structure so outputs
# can be compared directly between implementations.
# All variables come from prior cells — no new Spark jobs.

from datetime import datetime

# ── Per-rule breakdown from the already-cached brd_matches ───────────────
# One single Spark job to get counts per rule description.
rule_breakdown = (
    brd_matches
    .groupBy("priority", "category", "description")
    .count()
    .orderBy("priority")
    .collect()
)

# ── Match rate calculations (based on UNIQUE axis, not rows) ────────────
brd_match_rate_report    = brd_unique_axis / ORIGINAL_AXIS_COUNT * 100
greedy_match_rate_report = total_greedy    / ORIGINAL_AXIS_COUNT * 100
final_match_rate_report  = total_matched   / ORIGINAL_AXIS_COUNT * 100
unmatched_count_report   = ORIGINAL_AXIS_COUNT - total_matched

# ── Build report string ──────────────────────────────────────────────────
rule_lines = "\n".join(
    f"  [{row['category']:6s} P{row['priority']:02d}] {row['description'][:55]:<55s} : {row['count']:>8,}"
    for row in rule_breakdown
)

report = f"""
{'='*80}
DERIVATIVES MATCHING — ACCURACY REPORT  (BRD + GREEDY)
PySpark / Databricks  |  Run: {RUN_ID[:8]}...  |  Batch: {BATCH_ID}
{'='*80}

Generated : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
Rule Ver  : {RULE_VERSION}
Scope     : {'SOPHIS/DELTA1 EXCLUDED' if EXCLUDE_SOPHIS_DELTA1 else 'ALL SYSTEMS'}

{'='*80}
INPUT DATA
{'='*80}
  Total Axis (original) : {ORIGINAL_AXIS_COUNT_FULL:>12,}
  Total Finstore        : {ORIGINAL_FINSTORE_COUNT:>12,}
  Excluded (scope)      : {excluded_count:>12,}
  Axis (in-scope)       : {ORIGINAL_AXIS_COUNT:>12,}

{'='*80}
LAYER 1: BRD DETERMINISTIC MATCHING  (15 rules: 3 SOPHIS + 10 OTC + 2 ETD)
{'='*80}
{rule_lines}
{'─'*80}
  BRD match rows (w/r m:m): {brd_match_rows:>12,}
  Unique Axis matched   : {brd_unique_axis:>12,}
  Unique Finstore used  : {brd_unique_fin:>12,}
  BRD Match Rate (axis) : {brd_match_rate_report:>11.2f}%

{'='*80}
LAYER 2: GREEDY / PROBABILISTIC MATCHING
{'='*80}
  Strategy 1  Amount + Counterparty ({GREEDY_TOLERANCE_PCT*100:.1f}%) : {greedy1_count:>8,}
  Strategy 2  Amount Strict         ({STRICT_TOLERANCE_PCT*100:.1f}%) : {greedy2_count:>8,}
{'─'*80}
  Total Greedy matched  : {total_greedy:>12,}
  Greedy Match Rate     : {greedy_match_rate_report:>11.2f}%

{'='*80}
FINAL COMBINED RESULTS
{'='*80}
  BRD (unique axis)     : {brd_unique_axis:>12,}  ({brd_match_rate_report:.2f}%)
  Greedy                : {total_greedy:>12,}  ({greedy_match_rate_report:.2f}%)
{'─'*80}
  TOTAL UNIQUE MATCHED  : {total_matched:>12,}
  TOTAL MATCH ROWS      : {total_match_rows:>12,}
  TOTAL UNMATCHED (Axis): {unmatched_count_report:>12,}

  >>> FINAL MATCH RATE (unique axis): {final_match_rate_report:>10.2f}% <<<

{'='*80}
OUTPUT TABLES  (saved to {OUTPUT_DIR})
{'='*80}
  Narrow (base — before enrichment):
    - matched_all_base         ({total_match_rows:,} rows, ~15 cols)
    - matched_brd_layer        ({brd_match_rows:,} rows)
    - matched_greedy_layer     ({total_greedy:,} rows)
    - unmatched_axis_base
    - unmatched_finstore_base

  Wide (enriched — 100+ cols):
    - matched_all_enriched     ({total_match_rows:,} rows)
    - unmatched_axis_enriched
    - unmatched_finstore_enriched

{'='*80}
MATCHING SEMANTICS  (v4.7 — Waterfall Parity)
{'='*80}
  BRD: best rule per axis, all fin matches within rule    : Pandas waterfall
  Greedy layers: 1 axis → 1 best fin (window dedup)      : Pandas nsmallest
  Pool removal uses DISTINCT axis/fin IDs                 : Pandas set.unique
  Match rate based on UNIQUE axis count                   : Pandas .nunique()
  Runtime assertions verified in Section 15a

{'='*80}
"""

print(report)

# ── Save report ──────────────────────────────────────────────────────────
try:
    # Databricks
    dbutils.fs.put(f"{OUTPUT_DIR}/accuracy_report.txt", report, overwrite=True)
    print(f"✅ Accuracy report saved via dbutils to: {OUTPUT_DIR}/accuracy_report.txt")
except NameError:
    # Local / non-Databricks fallback
    import os
    report_path = OUTPUT_DIR.replace("dbfs:", "/dbfs") if OUTPUT_DIR.startswith("dbfs:") else OUTPUT_DIR
    os.makedirs(report_path, exist_ok=True)
    out_file = os.path.join(report_path, "accuracy_report.txt")
    with open(out_file, "w") as f:
        f.write(report)
    print(f"✅ Accuracy report saved locally to: {out_file}")

---
## 🔹 Section 23: Verdict

Pass / fail assessment against the 85% match rate target.

In [ ]:
# ============================================================
# SECTION 23: VERDICT
# ============================================================
# Identical target thresholds as the Pandas reference notebook
# (85% = success, 80–84.99% = acceptable, <80% = fail).
# Uses the same final_match_rate_report variable computed in Section 22.

TARGET_RATE = 85.0
ACCEPTABLE_RATE = 80.0

print(f"\n{'='*80}")
print("VERDICT")
print(f"{'='*80}")
print(f"\n  Target           : {TARGET_RATE:.0f}%")
print(f"  Achieved         : {final_match_rate_report:.2f}%")
print(f"  BRD contribution : {brd_match_rate_report:.2f}%")
print(f"  Greedy contrib.  : {greedy_match_rate_report:.2f}%")
print(f"  Unmatched (Axis) : {unmatched_count_report:,}")
print()

if final_match_rate_report >= TARGET_RATE:
    verdict_label  = "✅  TARGET ACHIEVED"
    verdict_status = "SUCCESS"
    verdict_note   = f"Exceeded target by {final_match_rate_report - TARGET_RATE:.2f} percentage points."

elif final_match_rate_report >= ACCEPTABLE_RATE:
    verdict_label  = "⚠️   TARGET NEARLY ACHIEVED"
    verdict_status = f"ACCEPTABLE  (within {ACCEPTABLE_RATE:.0f}–{TARGET_RATE:.0f}% range)"
    verdict_note   = (
        f"Gap to target: {TARGET_RATE - final_match_rate_report:.2f} pp.  "
        f"Review unmatched_reason breakdown (Section 18) for remediation."
    )

else:
    verdict_label  = "❌  TARGET NOT ACHIEVED"
    verdict_status = "FAIL"
    verdict_note   = (
        f"Gap to target: {TARGET_RATE - final_match_rate_report:.2f} pp.  "
        f"Review unmatched_reason breakdown (Section 18) and consider "
        f"expanding greedy tolerances or adding new BRD rules."
    )

print(f"  {verdict_label}")
print(f"  Status : {verdict_status}")
print(f"  Note   : {verdict_note}")

print(f"\n{'='*80}")
print("Notebook execution complete.")
print(f"  Run ID   : {RUN_ID}")
print(f"  Batch ID : {BATCH_ID}")
print(f"{'='*80}")